# RTE student: DeBERTa-v3-xsmall, plain and distilled

Same code and pinned packages as the local venv and the teacher notebook. Produces:
1. `outputs/student_deberta_v3_xsmall/`: student trained on hard labels only (no-distillation comparison)
2. `outputs/distill_deberta_v3_xsmall/`: student distilled from the DeBERTa-v3-base teacher (the submission)

**Distillation needs the teacher logits** from `colab_teacher.ipynb`:
- Same runtime as the teacher notebook: they're already at `outputs/baseline_deberta_v3_base_oof/teacher_logits_train.npy`.
- New runtime: upload `teacher_artifacts.zip` in step 5.

The plain student doesn't need the teacher, so it can run before the teacher finishes.

Runtime → Change runtime type → **GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import sys; print(sys.version)

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 3 MiB
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


## 1. Get the code
Skips the clone if the repo is already here (e.g. same runtime as the teacher notebook).

In [ ]:
import os
REPO_URL = "https://github.com/ANONYMIZED/rte-challenge.git"

%cd /content
if not os.path.isdir("/content/rte-challenge"):
    if REPO_URL:
        !git clone -q $REPO_URL rte-challenge
    else:
        from google.colab import files
        up = files.upload()  # choose rte-challenge.zip
        !unzip -q -o {next(iter(up))} -d /content
%cd /content/rte-challenge
!git log --oneline -1 2>/dev/null || true

/content
/content/rte-challenge
62c9a47 (HEAD -> setup/project-scaffold, origin/setup/project-scaffold, origin/main, origin/HEAD) Resume interrupted runs, keep Colab outputs on Drive, recover metrics from logs


## 1b. Keep outputs on Google Drive
`outputs/` becomes a link to `MyDrive/rte-challenge/outputs`, so an expired runtime loses nothing. Training saves each finished seed and fold right away, and **re-running a training cell resumes**: finished seeds and folds are skipped.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/rte-challenge/outputs"
os.makedirs(DRIVE_OUT, exist_ok=True)
%cd /content/rte-challenge
if os.path.isdir("outputs") and not os.path.islink("outputs"):
    !cp -rn outputs/. "$DRIVE_OUT"/ && rm -rf outputs
if not os.path.islink("outputs"):
    os.symlink(DRIVE_OUT, "outputs")
!ls -la outputs/ outputs/*/ 2>/dev/null | head -40

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/rte-challenge
outputs/:
total 24
drwx------ 6 root root 4096 Nov 10 13:11 .
drwx------ 3 root root 4096 Nov 10 13:05 ..
drwx------ 2 root root 4096 Nov 10 04:51 baseline_deberta_v3_base
drwx------ 2 root root 4096 Nov 10 06:25 baseline_deberta_v3_base_oof
drwx------ 3 root root 4096 Nov 10 13:11 distill_deberta_v3_xsmall
drwx------ 2 root root 4096 Nov 10 13:08 student_deberta_v3_xsmall

outputs/baseline_deberta_v3_base/:
total 164
drwx------ 2 root root   4096 Nov 10 04:51 .
drwx------ 6 root root   4096 Nov 10 13:11 ..
-rw------- 1 root root    426 Nov 10 03:01 config.yaml
-rw------- 1 root root 121830 Nov 10 04:51 dev_predictions.jsonl
-rw------- 1 root root  11208 Nov 10 04:51 dev_probs_all_seeds.npy
-rw------- 1 root root  19442 Nov 10 03:01 environment.json
-rw------- 1 root root   5564 Nov 10 04:51 metrics.json
-rw------- 1 root root    109 No

## 2. Install pinned dependencies (keep Colab's CUDA torch)

In [ ]:
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!sed -i 's/<3.13/<3.14/' pyproject.toml
!pip install -q --no-deps -e .
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rte (pyproject.toml) ... done
2.14.1+cu130 5.22.2 True


## 3. Data
Skipped if `data/` already exists; otherwise upload the challenge zip.

In [ ]:
%cd /content/rte-challenge
if not (os.path.exists("data/train.jsonl") and os.path.exists("data/dev.jsonl")):
    from google.colab import files
    up = files.upload()
    ZIP = next(iter(up))
    !python scripts/prepare_data.py --zip "$ZIP" --out data
!wc -l data/train.jsonl data/dev.jsonl

/content/rte-challenge
   2490 data/train.jsonl
    277 data/dev.jsonl
   2767 total


## 4. Plain student: hard labels only, 5 seeds

In [ ]:
!python -m rte.train --config configs/student_deberta_v3_xsmall.yaml

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
run student_deberta_v3_xsmall | model microsoft/deberta-v3-xsmall | device cuda
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.

## 5. Teacher logits
Uses the logits from the teacher run if they're on this machine; otherwise upload `teacher_artifacts.zip`.

In [ ]:
LOGITS = "outputs/baseline_deberta_v3_base_oof/teacher_logits_train.npy"
if not os.path.exists(LOGITS):
    from google.colab import files
    up = files.upload()
    !unzip -q -o {next(iter(up))} -d /content/rte-challenge
import numpy as np
print(LOGITS, np.load(LOGITS).shape)  # expect (2490, 2)

outputs/baseline_deberta_v3_base_oof/teacher_logits_train.npy (2490, 2)


## 6. Distilled student: the submission, 5 seeds
Same hyperparameters as step 4; only the loss changes (alpha=0.5 CE + 0.5 KD at T=2).

In [ ]:
!python -m rte.distill --config configs/distill_deberta_v3_xsmall.yaml

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
run distill_deberta_v3_xsmall | model microsoft/deberta-v3-xsmall | device cuda
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddi

## 7. Compare and download
Unzip `student_artifacts.zip` in the local project root. The distilled checkpoint is kept (~280 MB) so you can re-run predictions locally; the plain student's is dropped.

In [ ]:
import json, glob
for f in sorted(glob.glob("outputs/*/metrics.json")):
    m = json.load(open(f))
    p = m["params"]
    print(f"{m['run_name']:32s} dev {m['dev_acc_mean']*100:5.1f} ± {m['dev_acc_std']*100:4.1f}  "
          f"params {p['total']/1e6:6.1f}M (non-emb {p['non_embedding']/1e6:5.1f}M)")

!zip -q -r student_artifacts.zip outputs/student_deberta_v3_xsmall outputs/distill_deberta_v3_xsmall -x 'outputs/student_deberta_v3_xsmall/model/*'
!du -h student_artifacts.zip
from google.colab import files
files.download("student_artifacts.zip")

baseline_deberta_v3_base         dev  84.8 ±  1.1  params  184.4M (non-emb  86.0M)
distill_deberta_v3_xsmall        dev  76.8 ±  1.6  params   70.8M (non-emb  21.6M)
student_deberta_v3_xsmall        dev  77.7 ±  2.1  params   70.8M (non-emb  21.6M)
203M	student_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>